# **TIỀN XỬ LÝ DỮ LIỆU**

---

**Mục tiêu:** Làm sạch và chuẩn hóa dữ liệu để phục vụ phân tích

**Thành viên thực hiện:** [Tên - MSSV]

## **1. Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os

import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11

## **2. Load dữ liệu thô**

In [ ]:
# Đọc dữ liệu
df = pd.read_csv('../data/raw/products_raw.csv')

print(f"Kích thước dữ liệu: {df.shape}")
print(f"Số dòng: {df.shape[0]:,}")
print(f"Số cột: {df.shape[1]}")

df.head()

## **3. Kiểm tra tính nhất quán dữ liệu**

### **3.1 Kiểm tra kiểu dữ liệu**

In [ ]:
print("Kiểu dữ liệu của các cột:")
df.info()

In [ ]:
# Chuyển đổi kiểu dữ liệu nếu cần
# Ví dụ:
# df['price'] = pd.to_numeric(df['price'], errors='coerce')
# df['crawl_date'] = pd.to_datetime(df['crawl_date'])

### **3.2 Kiểm tra giá trị thiếu (Missing Values)**

In [ ]:
# Thống kê missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Cột': df.columns,
    'Số lượng thiếu': missing.values,
    'Tỷ lệ (%)': missing_pct.values
})

missing_df[missing_df['Số lượng thiếu'] > 0].sort_values('Số lượng thiếu', ascending=False)

In [ ]:
# Visualize missing values
plt.figure(figsize=(12, 6))
missing_cols = missing_df[missing_df['Số lượng thiếu'] > 0].sort_values('Tỷ lệ (%)', ascending=False)

if len(missing_cols) > 0:
    plt.barh(missing_cols['Cột'], missing_cols['Tỷ lệ (%)'], color='coral')
    plt.xlabel('Tỷ lệ thiếu (%)')
    plt.title('Tỷ lệ giá trị thiếu theo cột', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("✓ Không có giá trị thiếu trong dữ liệu")

### **3.3 Xử lý giá trị thiếu**

In [ ]:
# Chiến lược xử lý:
# 1. Loại bỏ cột có quá nhiều giá trị thiếu (>50%)
# 2. Điền giá trị trung bình/trung vị cho biến số
# 3. Điền mode hoặc 'Unknown' cho biến phân loại
# 4. Loại bỏ dòng nếu thiếu thông tin quan trọng

# Ví dụ:
# df['rating'].fillna(df['rating'].median(), inplace=True)
# df['brand'].fillna('Unknown', inplace=True)
# df.dropna(subset=['price', 'product_name'], inplace=True)

print(f"Kích thước sau xử lý missing: {df.shape}")

### **3.4 Kiểm tra và xử lý Duplicate**

In [ ]:
# Kiểm tra duplicate
duplicates = df.duplicated().sum()
print(f"Số dòng trùng lặp hoàn toàn: {duplicates}")

# Kiểm tra duplicate theo product_id
if 'product_id' in df.columns:
    id_duplicates = df.duplicated(subset=['product_id']).sum()
    print(f"Số sản phẩm trùng ID: {id_duplicates}")
    
    if id_duplicates > 0:
        df = df.drop_duplicates(subset=['product_id'], keep='first')
        print(f"✓ Đã loại bỏ {id_duplicates} sản phẩm trùng lặp")

print(f"Kích thước sau xử lý duplicate: {df.shape}")

## **4. Phát hiện và xử lý Outliers**

### **4.1 Phát hiện Outliers bằng IQR**

In [ ]:
def detect_outliers_iqr(df, column):
    """
    Phát hiện outliers bằng phương pháp IQR
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    
    return outliers, lower_bound, upper_bound


# Kiểm tra outliers cho các cột số
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    if len(outliers) > 0:
        print(f"\n{col}:")
        print(f"  - Số outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
        print(f"  - Ngưỡng: [{lower:.2f}, {upper:.2f}]")

### **4.2 Visualize Outliers**

In [ ]:
# Boxplot để quan sát outliers
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols[:6]):
    axes[idx].boxplot(df[col].dropna())
    axes[idx].set_title(col, fontweight='bold')
    axes[idx].set_ylabel('Giá trị')

plt.tight_layout()
plt.show()

### **4.3 Xử lý Outliers**

In [ ]:
# Chiến lược xử lý outliers:
# 1. Loại bỏ nếu là lỗi dữ liệu rõ ràng (giá = 0, số âm,...)
# 2. Capping (giới hạn ở ngưỡng IQR)
# 3. Giữ nguyên nếu là giá trị hợp lý (sản phẩm cao cấp)

# Ví dụ: Loại bỏ giá = 0 hoặc âm
# df = df[df['price'] > 0]

# Ví dụ: Capping
# Q1 = df['price'].quantile(0.25)
# Q3 = df['price'].quantile(0.75)
# IQR = Q3 - Q1
# df['price'] = df['price'].clip(lower=Q1-1.5*IQR, upper=Q3+1.5*IQR)

print(f"Kích thước sau xử lý outliers: {df.shape}")

## **5. Feature Engineering**

Tạo các biến mới phục vụ phân tích

In [ ]:
# Ví dụ các biến mới:

# 1. Tính discount_amount (số tiền giảm)
# if 'original_price' in df.columns and 'price' in df.columns:
#     df['discount_amount'] = df['original_price'] - df['price']

# 2. Phân loại giá (price_range)
# df['price_range'] = pd.cut(df['price'], 
#                             bins=[0, 500000, 2000000, 10000000, float('inf')],
#                             labels=['Rất rẻ', 'Trung bình', 'Cao', 'Rất cao'])

# 3. Tính revenue ước tính
# if 'price' in df.columns and 'sold' in df.columns:
#     df['estimated_revenue'] = df['price'] * df['sold']

# 4. Phân loại shop theo số lượng bán
# shop_sales = df.groupby('shop_id')['sold'].sum().reset_index()
# shop_sales['shop_tier'] = pd.qcut(shop_sales['sold'], q=3, labels=['Small', 'Medium', 'Large'])
# df = df.merge(shop_sales[['shop_id', 'shop_tier']], on='shop_id', how='left')

print("Các cột sau Feature Engineering:")
print(df.columns.tolist())

## **6. Chuẩn hóa dữ liệu**

In [ ]:
# Chuẩn hóa tên cột (lowercase, remove spaces)
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Chuẩn hóa text (strip, lowercase cho các cột text)
text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

print("✓ Đã chuẩn hóa dữ liệu")

## **7. Kiểm tra dữ liệu sau xử lý**

In [ ]:
print("Thông tin dữ liệu sau xử lý:")
df.info()

print("\n" + "="*60)
print("Thống kê mô tả:")
df.describe()

In [ ]:
# Kiểm tra lại missing values
print("Giá trị thiếu sau xử lý:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## **8. Lưu dữ liệu đã xử lý**

In [ ]:
# Tạo thư mục nếu chưa có
os.makedirs('../data/processed', exist_ok=True)

# Lưu file CSV
output_file = '../data/processed/products_cleaned.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✓ Đã lưu dữ liệu đã xử lý vào: {output_file}")
print(f"✓ Kích thước cuối cùng: {df.shape}")
print(f"✓ Số dòng: {len(df):,}")
print(f"✓ Số cột: {df.shape[1]}")

## **9. Tổng kết**

**Quy trình tiền xử lý đã thực hiện:**
1. ✓ Kiểm tra và chuyển đổi kiểu dữ liệu
2. ✓ Xử lý giá trị thiếu (Missing Values)
3. ✓ Loại bỏ duplicate
4. ✓ Phát hiện và xử lý outliers
5. ✓ Feature Engineering
6. ✓ Chuẩn hóa dữ liệu

**Kết quả:**
- Dữ liệu ban đầu: [Số dòng] x [Số cột]
- Dữ liệu sau xử lý: [Số dòng] x [Số cột]
- Tỷ lệ giữ lại: [%]

**Bước tiếp theo:**
- Notebook 03: Phân tích khám phá dữ liệu (EDA)
- Notebook 04: Xây dựng Dashboard